# HEST-1k Coverage95 Generalization Readiness

This notebook records the current leave-slide-out, leave-organ-out, and leave-cohort-out task readiness for coverage95 HistoOmniST experiments. It does not report trained generalization performance; it checks that split-specific task manifests and configs can be materialized without test leakage.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent

out_dir = ROOT / "results/hest1k_human_visium_expression/generalization_readiness"
summary = json.loads((out_dir / "run_summary.json").read_text(encoding="utf-8"))
tasks = pd.read_csv(out_dir / "task_summary.csv")
generated = pd.read_csv(out_dir / "generated_task_files.csv")
summary

{'expression_manifest': 'data/HEST-1k/manifests/human_visium_sf_manifest_highconf_context.csv',
 'split_types': ['leave_slide_out', 'leave_organ_out', 'leave_cohort_out'],
 'n_tasks': 28,
 'n_ready_tasks': 15,
 'n_generated_task_files': 7,
 'min_test_slides': 5,
 'val_fraction': 0.1,
 'outputs': {'task_summary': 'results/hest1k_human_visium_expression/generalization_readiness/task_summary.csv',
  'generated_task_files': 'results/hest1k_human_visium_expression/generalization_readiness/generated_task_files.csv'}}

In [2]:
by_type = (
    tasks.groupby("split_type")
    .agg(
        n_tasks=("heldout", "count"),
        n_ready=("ready_for_split_specific_training", "sum"),
        max_test_slides=("n_test_slides", "max"),
    )
    .reset_index()
)
assert summary["n_tasks"] == len(tasks)
assert summary["n_ready_tasks"] == int(tasks["ready_for_split_specific_training"].sum())
assert set(["leave_slide_out", "leave_organ_out", "leave_cohort_out"]).issubset(set(by_type["split_type"]))
by_type

,split_type,n_tasks,n_ready,max_test_slides
0,leave_cohort_out,19,9,59
1,leave_organ_out,8,5,73
2,leave_slide_out,1,1,48


In [3]:
generated_view = generated[
    [
        "split_type",
        "heldout",
        "n_train_slides",
        "n_val_slides",
        "n_test_slides",
        "task_manifest",
        "expression_config",
        "sf_config",
    ]
]
assert len(generated_view) == summary["n_generated_task_files"]
assert (generated_view["n_train_slides"] > 0).all()
assert (generated_view["n_val_slides"] > 0).all()
assert (generated_view["n_test_slides"] >= summary["min_test_slides"]).all()
generated_view

,split_type,heldout,n_train_slides,n_val_slides,n_test_slides,task_manifest,expression_config,sf_config
0,leave_slide_out,leave_slide_out,163,23,48,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
1,leave_organ_out,Bowel,145,16,73,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
2,leave_organ_out,Brain,173,19,42,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
3,leave_organ_out,Breast,203,23,8,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
4,leave_cohort_out,COLON MAP: Colon Molecular Atlas Project,174,19,41,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
5,leave_cohort_out,Charting the Heterogeneity of Colorectal Cance...,198,22,14,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...
6,leave_cohort_out,GENE EXPRESSION WITHIN A HUMAN CHOROIDAL NEOVA...,206,23,5,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...,results/hest1k_human_visium_expression/general...


In [4]:
breast = generated[generated["task_slug"].eq("breast")].iloc[0]
breast_manifest = pd.read_csv(ROOT / breast["task_manifest"])
breast_counts = breast_manifest["split"].value_counts().to_dict()
breast_check = {
    "task_slug": breast["task_slug"],
    "heldout": breast["heldout"],
    "manifest_rows": len(breast_manifest),
    "train": int(breast_counts.get("train", 0)),
    "val": int(breast_counts.get("val", 0)),
    "test": int(breast_counts.get("test", 0)),
    "test_organs": "|".join(sorted(breast_manifest.loc[breast_manifest["split"].eq("test"), "organ"].astype(str).unique())),
}
assert breast_check["train"] == int(breast["n_train_slides"])
assert breast_check["val"] == int(breast["n_val_slides"])
assert breast_check["test"] == int(breast["n_test_slides"])
assert breast_check["test_organs"] == "Breast"
breast_check

{'task_slug': 'breast',
 'heldout': 'Breast',
 'manifest_rows': 234,
 'train': 203,
 'val': 23,
 'test': 8,
 'test_organs': 'Breast'}